In [ ]:
# GPU + version checks

import torch, ultralytics

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU")
print("Ultralytics:", ultralytics.__version__)

In [ ]:
# model training

from ultralytics import YOLO

# model = YOLO("yolo11s.pt")

# # Fine-tune more using the last weights
model = YOLO("runs/detect/train/weights/last.pt") 

model.train(
    data="../dataset.yaml",
    epochs=30,
    imgsz=1024,
    batch=4,
    workers=4,
    project="runs/detect",
    name="train",
    exist_ok=True
)

In [ ]:
# resuming training from last checkpoint
from ultralytics import YOLO
model = YOLO("runs/detect/train/weights/last.pt") 
model.train(resume=True)

In [ ]:
# test trained model (make sure to put the best.pt file in "models/trained") 

from ultralytics import YOLO
model = YOLO("../models/best.pt")

model.predict(
    source="../data/images/predict/25.11.12(1)_page-0004.jpg", # test image name
    project="runs/detect",
    name="predict",
    conf=0.25,
    save=True,
    show_labels=False,
    # show_conf=False
)

In [3]:
from ultralytics import YOLO
import cv2
import os
import json

model = YOLO("../notebooks/runs/detect/train/weights/best.pt")

save_dir = "runs/detect/predict"
os.makedirs(save_dir, exist_ok=True)

results = model.predict(
    source=[
        "../data/images/predict/25.11.12(1)_page-0004.jpg",
        "../data/images/predict/25.10.31_page-0004.jpg",
        "../data/images/predict/25.10.13_page-0012.jpg",
        "../data/images/predict/TAKEOFF1_page-0002.jpg",
        "../data/images/predict/25.09.16_page-0007.jpg",
    ],
    stream=True,
    conf=0.250,
)

index = 0
all_detections = []  # store all detections here

for r in results:
    img = r.orig_img.copy()
    detections = []  # detections for this image

    for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
        x1, y1, x2, y2 = map(int, box)
        class_id = int(cls)
        class_name = model.names[class_id]
        confidence = float(conf)

        # Save detection info
        detections.append({
            "class_id": class_id,
            "class_name": class_name,
            "confidence": confidence,
            "bbox": [x1, y1, x2, y2]
        })

        # Draw box
        cv2.rectangle(img, (x1, y1), (x2, y2), (0,0,255), 2)
        cv2.putText(img, f"{class_name} {confidence:.2f}", (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 2)

    # Save annotated image
    out_path = f"{save_dir}/result_{index}.jpg"
    cv2.imwrite(out_path, img)

    # Add detections to full list
    all_detections.append({
        "image_index": index,
        "image_path": r.path,
        "detections": detections
    })

    index += 1

# save all detections to JSON
json_path = f"{save_dir}/detections.json"
with open(json_path, "w") as f:
    json.dump(all_detections, f, indent=4)

print("\n=== Detection Summary ===")
for img_result in all_detections:
    print(f"\nImage {img_result['image_index']}: {img_result['image_path']}")
    for d in img_result["detections"]:
        print(f"  {d['class_name']} ({d['confidence']:.2f}) → {d['bbox']}")


d:\Sudharsanan.M\Projects\Bartos\repo\floorplanner-cost-estimate\training_model\.venv\Lib\site-packages\PIL\Image.py:3432: DecompressionBombWarning: Image size (113400000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


0: 1024x1024 2 DOASs, 72 Supply Diffuser 24*24(A)s, 786.2ms
1: 1024x1024 (no detections), 786.2ms
2: 1024x1024 11 EX RTUs, 1 Fan EF, 35 Supply Diffuser 24*24(A)s, 786.2ms
3: 1024x1024 (no detections), 786.2ms
4: 1024x1024 (no detections), 786.2ms
Speed: 19.8ms preprocess, 786.2ms inference, 1.2ms postprocess per image at shape (1, 3, 1024, 1024)

=== Detection Summary ===

Image 0: ../data/images/predict/25.11.12(1)_page-0004.jpg
  Supply Diffuser 24*24(A) (0.62) → [7042, 7660, 7227, 7958]
  Supply Diffuser 24*24(A) (0.62) → [9315, 6013, 9469, 6774]
  Supply Diffuser 24*24(A) (0.58) → [1130, 3323, 1306, 3633]
  Supply Diffuser 24*24(A) (0.57) → [7639, 5103, 7846, 5390]
  Supply Diffuser 24*24(A) (0.55) → [4484, 7568, 4662, 7865]
  Supply Diffuser 24*24(A) (0.53) → [8526, 3321, 8745, 3617]
  Supply Diffuser 24*24(A) (0.53) → [6040, 7662, 6248, 7963]
  Supply Diffuser 24*24(A) (0.53) → [7041, 7753, 7236, 8042]
  Supply Diffuser 24*24(A) (0.53) → [9305, 3829, 9475, 4215]
  Supply Diffuser

In [ ]:
import cv2
import json
import os
# for redrawing saved detections using the json file
# Path to your saved JSON file
json_path = "runs/detect/predict/detections.json"

# Load saved detections
with open(json_path, "r") as f:
    all_detections = json.load(f)

# Output directory for redraw
save_dir = "runs/detect/redrawn"
os.makedirs(save_dir, exist_ok=True)

for item in all_detections:
    image_path = item["image_path"]
    detections = item["detections"]

    # Load original image
    img = cv2.imread(image_path)

    if img is None:
        print(f"Cannot load image: {image_path}")
        continue

    # Draw each saved bounding box
    for det in detections:
        x1, y1, x2, y2 = det["bbox"]
        class_name = det["class_name"]
        conf = det["confidence"]

        # Draw rectangle
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)

        # Label text
        label = f"{class_name} {conf:.2f}"
        cv2.putText(img, label, (x1, y1 - 5),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    # Save re-drawn image
    out_path = os.path.join(save_dir, f"redrawn_{item['image_index']}.jpg")
    cv2.imwrite(out_path, img)

    print(f"Saved: {out_path}")
